# Pricing and Margin Analysis

This notebook analyzes pricing strategies and profit margins
to optimize revenue and profitability across product categories.

Key goals:

• Evaluate current pricing effectiveness
• Analyze margin distribution by category
• Identify opportunities for price optimization

In [ ]:
import sys
import os

# Add project root to Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
# Load data
sessions = pd.read_csv("../data/raw/sessions.csv")
orders = pd.read_csv("../data/raw/orders.csv")
customers = pd.read_csv("../data/raw/customers.csv")

# Merge data
customer_sessions = sessions.groupby('customer_id').agg({
    'session_id': 'count',
    'session_duration': 'mean',
    'pages_viewed': 'mean',
    'bounced_flag': 'mean'
}).reset_index()

customer_orders = orders.groupby('customer_id').agg({
    'order_id': 'count',
    'final_price': 'sum',
    'margin': 'sum',
    'discount_pct': 'mean'
}).reset_index()

# Merge all
customer_data = customers.merge(customer_sessions, on='customer_id', how='left').merge(customer_orders, on='customer_id', how='left')

# Fill NaN
customer_data = customer_data.fillna(0)

customer_data.head()

In [ ]:
# Define churn: customers with no orders in last 30 days (simulated)
# For demo, randomly assign churn based on engagement
np.random.seed(42)
customer_data['churn'] = np.random.choice([0, 1], size=len(customer_data), p=[0.7, 0.3])

# But make it realistic: low engagement -> higher churn
customer_data['engagement_score'] = (
    customer_data['session_id'] * 0.3 +
    customer_data['session_duration'] * 0.001 +
    customer_data['pages_viewed'] * 0.2 +
    customer_data['order_id'] * 0.5
)

customer_data['churn'] = (customer_data['engagement_score'] < customer_data['engagement_score'].median()).astype(int)

customer_data['churn'].value_counts()

In [ ]:
# Prepare features
features = ['acquisition_channel', 'session_id', 'session_duration', 'pages_viewed', 'bounced_flag', 
           'order_id', 'final_price', 'margin', 'discount_pct']

X = customer_data[features]
y = customer_data['churn']

# Encode categorical
le = LabelEncoder()
X['acquisition_channel'] = le.fit_transform(X['acquisition_channel'])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Feature Importance for Churn Prediction')
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()